# SMA Module 3 — Time Series Modeling

Month/week dummies, two extra hand rules, and a decision tree. Run cells from top to bottom. The parquet stays on E:.


## 0) Imports


In [35]:
import os
# keep caches on E: — C: fills up fast on this machine
os.environ.setdefault("MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
os.environ.setdefault("TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("TMP", r"E:\IT_SPACES\AI\.cache\tmp")

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score
from sklearn.tree import DecisionTreeClassifier


## 1) Lesson dataframe

Same file as the Module 3 lesson. It is already next to that notebook, so this folder does not download another copy. I read that local parquet, not a Colab `/content` path.


In [36]:
data_path = Path(r"E:\IT_SPACES\AI\ZoomCamp\SMA\03\modeling\stocks_df_combined_2026_09_18.parquet.brotli")
df_full = pd.read_parquet(data_path)
print(df_full.shape, data_path.stat().st_size)
# the lesson list says Close and Adj Close_x; this file has Close_x
print("Close_x" in df_full.columns, "Close" in df_full.columns)


(240759, 203) 143841500
True False


## 2) Features, week-of-month, split


In [37]:
GROWTH = [g for g in df_full.keys() if (g.find("growth_") == 0) and (g.find("future") < 0)]
OHLCV = ["Open", "High", "Low", "Close", "Adj Close_x", "Volume"]
TO_PREDICT = [g for g in df_full.keys() if (g.find("future") >= 0)]
CUSTOM_NUMERICAL = [
    "SMA10",
    "SMA20",
    "growing_moving_average",
    "high_minus_low_relative",
    "volatility",
    "ln_volume",
]
TECHNICAL_INDICATORS = [
    "adx", "adxr", "apo", "aroon_1", "aroon_2", "aroonosc",
    "bop", "cci", "cmo", "dx", "macd", "macdsignal", "macdhist", "macd_ext",
    "macdsignal_ext", "macdhist_ext", "macd_fix", "macdsignal_fix",
    "macdhist_fix", "mfi", "minus_di", "mom", "plus_di", "dm", "ppo",
    "roc", "rocp", "rocr", "rocr100", "rsi", "slowk", "slowd", "fastk",
    "fastd", "fastk_rsi", "fastd_rsi", "trix", "ultosc", "willr",
    "ad", "adosc", "obv", "atr", "natr", "ht_dcperiod", "ht_dcphase",
    "ht_phasor_inphase", "ht_phasor_quadrature", "ht_sine_sine", "ht_sine_leadsine",
    "ht_trendmod", "avgprice", "medprice", "typprice", "wclprice",
]
TECHNICAL_PATTERNS = [g for g in df_full.keys() if g.find("cdl") >= 0]
MACRO = [
    "gdppot_us_yoy", "gdppot_us_qoq", "cpi_core_yoy", "cpi_core_mom", "FEDFUNDS",
    "DGS1", "DGS5", "DGS10",
]

# volume 0 would make log infinite, so those rows become a tiny positive number first
df_full["ln_volume"] = df_full.Volume.replace(0, np.nan).fillna(1e-9).apply(lambda x: np.log(x))
NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO
missing_ohlcv = [c for c in OHLCV if c not in df_full.columns]
print("missing ohlcv", missing_ohlcv)

df = df_full[df_full.Date >= "2000-01-01"].copy()
df["Weekday"] = df["Weekday"].astype(str)
# Month is a month-start timestamp, so I built the week label from Date, and Month needs .dt.month before get_dummies
df["month_wom"] = (
    df["Date"].dt.month_name()
    + "_w"
    + ((df["Date"].dt.day - 1) // 7 + 1).astype(str)
)
df["Month"] = df["Month"].dt.month.astype(str)

CATEGORICAL = ["Month", "Weekday", "Ticker", "ticker_type", "month_wom"]
dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype="int32")
DUMMIES = dummy_variables.keys().to_list()
df_with_dummies = pd.concat([df, dummy_variables], axis=1)
print("dummies", len(DUMMIES), "month_wom", sum(c.startswith("month_wom_") for c in DUMMIES))
print(df.shape)


missing ohlcv ['Close', 'Adj Close_x']
dummies 115 month_wom 60
(202292, 205)


In [38]:
def temporal_split(df, min_date, max_date, train_prop=0.7, val_prop=0.15, test_prop=0.15):
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)
    split_labels = []
    for date in df["Date"]:
        if date <= train_end:
            split_labels.append("train")
        elif date <= val_end:
            split_labels.append("validation")
        else:
            split_labels.append("test")
    df = df.copy()
    df["split"] = split_labels
    return df


new_df = temporal_split(df_with_dummies, df_with_dummies.Date.min(), df_with_dummies.Date.max())
print(new_df["split"].value_counts().to_dict())


{'train': 136842, 'test': 33163, 'validation': 32287}


## Question 1: Dummies for Month and Week-of-Month

Absolute correlation of the strongest `month_wom` dummy with `is_positive_growth_30d_future`, to three decimals.


In [39]:
target = "is_positive_growth_30d_future"
wom_cols = [c for c in DUMMIES if c.startswith("month_wom_")]
corrs = new_df[wom_cols + [target]].corr()[target].drop(target)
corr_df = corrs.rename("corr").to_frame()
corr_df["abs_corr"] = corr_df["corr"].abs()
corr_df = corr_df.sort_values("abs_corr", ascending=False)
print(corr_df.head(5))
# get_dummies prefixes the column, so the name is month_wom_October_w4, not October_w4
print("Q1", round(float(corr_df.iloc[0]["abs_corr"]), 3), corr_df.index[0])


                           corr  abs_corr
month_wom_October_w4   0.024584  0.024584
month_wom_November_w3  0.023251  0.023251
month_wom_February_w1 -0.020861  0.020861
month_wom_January_w2  -0.019846  0.019846
month_wom_January_w5  -0.019792  0.019792
Q1 0.025 month_wom_October_w4


## Question 2: New hand rules

Precision of the better rule, `pred3` or `pred4`, to three decimals.


In [40]:
new_df["pred0_manual_cci"] = (new_df.cci > 200).astype(int)
new_df["pred1_manual_prev_g1"] = (new_df.growth_30d > 1).astype(int)
new_df["pred2_manual_prev_g1_and_snp"] = (
    (new_df["growth_30d"] > 1) & (new_df["growth_snp500_30d"] > 1)
).astype(int)
new_df["pred3_manual_dgs10_5"] = ((new_df["DGS10"] <= 4.5) & (new_df["DGS5"] <= 4)).astype(int)
new_df["pred4_manual_dgs10_fedfunds"] = (
    (new_df["DGS10"] > 4) & (new_df["FEDFUNDS"] <= 4.795)
).astype(int)

test = new_df["split"] == "test"
y_test = new_df.loc[test, target]


def precision_on_test(pred_col):
    pred = new_df.loc[test, pred_col]
    score = precision_score(y_test.astype(int), pred.astype(int), zero_division=0)
    n_pos = int((pred == 1).sum())
    return float(score), n_pos


for col in [
    "pred0_manual_cci",
    "pred1_manual_prev_g1",
    "pred2_manual_prev_g1_and_snp",
    "pred3_manual_dgs10_5",
    "pred4_manual_dgs10_fedfunds",
]:
    score, n_pos = precision_on_test(col)
    print(col, round(score, 3), score, n_pos)


pred0_manual_cci 0.565 0.5646036916395223 921
pred1_manual_prev_g1 0.579 0.5786616557051885 19158
pred2_manual_prev_g1_and_snp 0.573 0.5725531356599318 15244
pred3_manual_dgs10_5 0.588 0.5877435575109994 15910
pred4_manual_dgs10_fedfunds 0.503 0.5025438100621821 15921


## Question 3: Unique correct predictions from a depth-10 tree

How many TEST rows are right only for `pred5_clf_10`?


In [41]:
features_list = NUMERICAL + DUMMIES
X_all = new_df[features_list].replace([np.inf, -np.inf], np.nan).fillna(0)
y_all = new_df[target]
train_mask = new_df["split"].isin(["train", "validation"])
print("features", X_all.shape[1], "train rows", int(train_mask.sum()), "missing y", int(y_all.isna().sum()))

clf10 = DecisionTreeClassifier(max_depth=10, random_state=42)
clf10.fit(X_all.loc[train_mask], y_all.loc[train_mask].astype(int))
# the lesson merge looks up prediction columns that are not on the frame yet, so I wrote the vector in the same row order
new_df["pred5_clf_10"] = clf10.predict(X_all)

hand = [
    "pred0_manual_cci",
    "pred1_manual_prev_g1",
    "pred2_manual_prev_g1_and_snp",
    "pred3_manual_dgs10_5",
    "pred4_manual_dgs10_fedfunds",
]
only_pred5 = new_df["pred5_clf_10"] == y_all
for col in hand:
    only_pred5 = only_pred5 & (new_df[col] != y_all)
new_df["only_pred5_is_correct"] = only_pred5.astype(int)
print("Q3", int(new_df.loc[test, "only_pred5_is_correct"].sum()))
print("pred5", precision_on_test("pred5_clf_10"))


features 299 train rows 169129 missing y 0
Q3 1243
pred5 (0.5912146499768196, 8628)


On my run, pred5 matches the label and every hand rule misses on 1243 test rows. 1187 of those labels are 1 and 56 are 0. pred5 is a true positive and no hand rule predicted 1 on 1187 test rows. With the tree correct, exactly 0, 1, 2, 3, 4, and 5 hand rules wrong comes to 613, 2680, 3905, 4875, 2515, and 1243. The form choices are 271, 571, 1171, and 1571. 1243 is not a choice. The nearest is 1171.


## Question 4: Best tree depth

Depth from 1 to 12 with the highest precision on TEST.

Depth 1 called every test row positive. The highest test precision was at depth 4.


In [42]:
best_depth = None
best_prec = -1.0
best_clf = None
for depth in range(1, 13):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_all.loc[train_mask], y_all.loc[train_mask].astype(int))
    pred_test = pd.Series(clf.predict(X_all.loc[test]), index=y_test.index)
    prec = float(precision_score(y_test.astype(int), pred_test.astype(int), zero_division=0))
    n_pos = int((pred_test == 1).sum())
    print(f"depth {depth} precision {prec:.6f} n_pos {n_pos}")
    if prec > best_prec:
        best_prec = prec
        best_depth = depth
        best_clf = clf

new_df["pred6_clf_best"] = best_clf.predict(X_all)
print("Q4", best_depth, round(best_prec, 3), best_prec)
print("pred6", precision_on_test("pred6_clf_best"))


depth 1 precision 0.570093 n_pos 33163
depth 2 precision 0.622534 n_pos 11760
depth 3 precision 0.571969 n_pos 14263
depth 4 precision 0.629175 n_pos 6677
depth 5 precision 0.599973 n_pos 7487
depth 6 precision 0.606205 n_pos 6704
depth 7 precision 0.621072 n_pos 6682
depth 8 precision 0.621785 n_pos 9410
depth 9 precision 0.611179 n_pos 9053
depth 10 precision 0.591215 n_pos 8628
depth 11 precision 0.602754 n_pos 7552
depth 12 precision 0.581944 n_pos 8518
Q4 4 0.629 0.6291747790924068
pred6 (0.6291747790924068, 6677)


## Question 5: What data is missing?

The depth-10 tree leans on US core CPI, one-year gold and oil growth, and the funds rate. I would add the CPI surprise (actual minus consensus), because the file already has the CPI change and not whether the print was a surprise. A public source is the Cleveland Fed inflation nowcast.
